# Notebook 01 — Recolección y Exploración de Datos de Precios

---

## TFM: Integración de Algoritmos de Aprendizaje Profundo para el Análisis de Sentimiento en Mercados Financieros

**Universidad Internacional de La Rioja (UNIR)**  
**Master Universitario en Análisis y Visualización de Datos Masivos**  
**Autores:** Armas Silva Stiv Alexis | Armas Silva Jonathan Fernando | Requelme Campoverde Adrian Alexander  
**Director:** Cigales Canga Jesús

---

## Objetivo de este Notebook

Este primer notebook cubre la **Fase 1: Recolección y Exploración Inicial de Datos** del proyecto.

Al finalizar este notebook habremos:

1. Descargado precios históricos de activos financieros relevantes para el contexto ecuatoriano
2. Explorado estadísticamente los datos de precios
3. Visualizado las series temporales y detectado patrones clave
4. Analizado la correlación entre los diferentes activos
5. Preparado el dataset maestro para las siguientes fases del pipeline

---

## Marco Teórico — ¿Por qué estos activos?

Ecuador presenta características únicas como mercado financiero:

- **Economía dolarizada**: No tiene moneda propia, usa el USD desde el año 2000. Esto hace que la política monetaria local sea limitada y los mercados internacionales tengan influencia directa.
- **Alta adopción de criptomonedas**: Ante la falta de soberanía monetaria, Bitcoin y Ethereum tienen alta penetración como activos alternativos de ahorro.
- **Dependencia de commodities**: El petróleo, banano y camarón son los pilares de exportación. Sus precios internacionales afectan directamente la economía del país.
- **Mercado bursátil pequeño**: La BVQ y BVG tienen baja liquidez comparada con mercados regionales.

Por estas razones, nuestro análisis se centra en activos con alta sensibilidad al sentimiento del mercado:
- **BTC y ETH**: Alta correlación con el sentimiento en redes sociales en español latinoamericano
- **SPY y EEM**: Referencia de los mercados globales y emergentes
- **ILF**: ETF de Latinoamérica directamente vinculado a la región de Ecuador
- **GLD y USO**: Oro y Petróleo como commodities clave para la economía ecuatoriana

---
## Sección 1: Configuración del Entorno

Cargamos todas las librerías necesarias y configuramos los parámetros globales.

In [1]:
# ============================================================
# CELDA 1: Importación de librerías y configuración
# ============================================================
# pandas/numpy  -> manipulación de datos numéricos
# matplotlib    -> visualización estática
# seaborn       -> visualización estadística avanzada
# scipy         -> estadística (test de normalidad)
# pathlib       -> manejo de rutas de archivos multiplataforma

import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Backend sin pantalla (para ejecucion automatica)
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from pathlib import Path
from datetime import datetime

# ── Configuración de rutas ─────────────────────────────────
# Detectamos la raiz del proyecto independientemente de donde
# se ejecute el notebook (desde notebooks/ o desde la raiz)
NOTEBOOK_DIR = Path(os.path.abspath(''))
if NOTEBOOK_DIR.name == 'notebooks':
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

PRICES_DIR  = PROJECT_ROOT / 'data' / 'raw' / 'prices'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ── Configuración de gráficos ──────────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)

print('Librerias cargadas correctamente')
print(f'Raiz del proyecto : {PROJECT_ROOT}')
print(f'Directorio precios: {PRICES_DIR}')
print(f'Directorio figuras: {FIGURES_DIR}')

Librerias cargadas correctamente
Raiz del proyecto : C:\Users\StivArmas\OneDrive - ROBALINO\Documentos\MASTER\TFM\tfm-sentiment-financial
Directorio precios: C:\Users\StivArmas\OneDrive - ROBALINO\Documentos\MASTER\TFM\tfm-sentiment-financial\data\raw\prices
Directorio figuras: C:\Users\StivArmas\OneDrive - ROBALINO\Documentos\MASTER\TFM\tfm-sentiment-financial\reports\figures


---
## Sección 2: Carga de Datos de Precios

Los datos fueron descargados en la Fase 1 y se encuentran en formato CSV.
Cada archivo contiene columnas **OHLCV**:
- **Open** : Precio de apertura del día
- **High** : Precio máximo del día
- **Low**  : Precio mínimo del día
- **Close**: Precio de cierre del día ← el más usado en análisis
- **Volume**: Volumen de transacciones del día

In [2]:
# ============================================================
# CELDA 2: Definición de activos y carga de archivos CSV
# ============================================================

# Catálogo de activos con metadata descriptiva para el análisis
ACTIVOS = {
    'BTC-USD': {'file': 'BTC_USD_1d.csv',  'nombre': 'Bitcoin',               'tipo': 'Criptomoneda', 'color': '#F7931A'},
    'ETH-USD': {'file': 'ETH_USD_1d.csv',  'nombre': 'Ethereum',              'tipo': 'Criptomoneda', 'color': '#627EEA'},
    'SPY':     {'file': 'SPY_1d.csv',      'nombre': 'S&P 500 ETF',           'tipo': 'ETF Global',   'color': '#2196F3'},
    'EEM':     {'file': 'EEM_1d.csv',      'nombre': 'Mercados Emergentes',   'tipo': 'ETF Regional', 'color': '#4CAF50'},
    'ILF':     {'file': 'ILF_1d.csv',      'nombre': 'Latinoamerica 40 ETF', 'tipo': 'ETF Latam',    'color': '#FF9800'},
    'EWZ':     {'file': 'EWZ_1d.csv',      'nombre': 'Brasil ETF',            'tipo': 'ETF Brasil',   'color': '#009C3B'},
    'GLD':     {'file': 'GLD_1d.csv',      'nombre': 'Oro (Gold ETF)',        'tipo': 'Commodity',    'color': '#FFD700'},
    'USO':     {'file': 'USO_1d.csv',      'nombre': 'Petroleo (Oil ETF)',    'tipo': 'Commodity',    'color': '#795548'},
}

# Cargar todos los archivos CSV en un diccionario de DataFrames
data = {}

print('Cargando datos de precios...')
print('-' * 75)

for ticker, info in ACTIVOS.items():
    filepath = PRICES_DIR / info['file']
    if filepath.exists():
        df = pd.read_csv(filepath, index_col=0, parse_dates=True)
        # Mantener solo columnas OHLCV
        cols_ohlcv = [c for c in ['Open', 'High', 'Low', 'Close', 'Volume'] if c in df.columns]
        data[ticker] = df[cols_ohlcv].copy()
        fecha_ini = df.index.min().date()
        fecha_fin = df.index.max().date()
        print(f'  OK  {ticker:8s} | {info["nombre"]:30s} | {len(df):,} dias | {fecha_ini} -> {fecha_fin}')
    else:
        print(f'  NO  {ticker:8s} | archivo no encontrado en: {filepath}')

print('-' * 75)
print(f'Total activos cargados: {len(data)} de {len(ACTIVOS)}')

Cargando datos de precios...
---------------------------------------------------------------------------
  OK  BTC-USD  | Bitcoin                        | 1,632 dias | 2022-01-01 -> 2026-06-20
  OK  ETH-USD  | Ethereum                       | 1,632 dias | 2022-01-01 -> 2026-06-20
  OK  SPY      | S&P 500 ETF                    | 1,119 dias | 2022-01-03 -> 2026-06-18
  OK  EEM      | Mercados Emergentes            | 1,119 dias | 2022-01-03 -> 2026-06-18
  OK  ILF      | Latinoamerica 40 ETF           | 1,119 dias | 2022-01-03 -> 2026-06-18
  OK  EWZ      | Brasil ETF                     | 1,119 dias | 2022-01-03 -> 2026-06-18
  OK  GLD      | Oro (Gold ETF)                 | 1,119 dias | 2022-01-03 -> 2026-06-18
  OK  USO      | Petroleo (Oil ETF)             | 1,119 dias | 2022-01-03 -> 2026-06-18
---------------------------------------------------------------------------
Total activos cargados: 8 de 8


In [3]:
# ============================================================
# CELDA 3: Inspección inicial — Bitcoin como caso de estudio
# ============================================================
# Examinamos Bitcoin como activo principal y más relevante
# para el análisis de sentimiento en contexto Ecuador

print('=' * 60)
print('INSPECCION DEL DATASET: Bitcoin (BTC-USD)')
print('=' * 60)

btc = data['BTC-USD']

print(f'\nDimensiones   : {btc.shape[0]:,} filas x {btc.shape[1]} columnas')
print(f'Periodo       : {btc.index.min().strftime("%d/%m/%Y")} al {btc.index.max().strftime("%d/%m/%Y")}')
print(f'Dias totales  : {(btc.index.max() - btc.index.min()).days:,}')
print(f'Valores nulos : {btc.isnull().sum().sum()}')

print('\n--- Primeras 5 filas ---')
display(btc.head())

print('\n--- Ultimas 5 filas ---')
display(btc.tail())

print('\n--- Estadisticas descriptivas ---')
display(btc.describe().round(2))

INSPECCION DEL DATASET: Bitcoin (BTC-USD)

Dimensiones   : 1,632 filas x 5 columnas
Periodo       : 01/01/2022 al 20/06/2026
Dias totales  : 1,631
Valores nulos : 0

--- Primeras 5 filas ---


,Open,High,Low,Close,Volume
Date,,,,,
2022-01-01,46311.7461,47827.3125,46288.4844,47686.8125,24582667004
2022-01-02,47680.9258,47881.4062,46856.9375,47345.2188,27951569547
2022-01-03,47343.5430,47510.7266,45835.9648,46458.1172,33071628362
2022-01-04,46458.8516,47406.5469,45752.4648,45897.5742,42494677905
2022-01-05,45899.3594,46929.0469,42798.2227,43569.0039,36851084859



--- Ultimas 5 filas ---


,Open,High,Low,Close,Volume
Date,,,,,
2026-06-16,66289.4609,66928.6094,65315.0703,65600.6406,25063963967
2026-06-17,65600.4297,66354.6250,63870.9297,64418.4453,31288885189
2026-06-18,64419.6445,64736.9219,62201.1445,62896.4727,30432454936
2026-06-19,62897.5156,63568.2227,62275.5820,63540.8359,22361931660
2026-06-20,63535.8086,64307.1367,63149.6602,64239.6758,17486384117



--- Estadisticas descriptivas ---


,Open,High,Low,Close,Volume
count,1632.0000,1632.0000,1632.0000,1632.0000,1632.0000
mean,58116.1900,59127.1900,57050.3700,58127.4700,35139435496.4000
std,31265.3300,31721.5400,30760.6900,31264.3100,21429131282.7900
min,15782.3000,16253.0500,15599.0500,15787.2800,5331172801.0000
25%,28320.9100,28744.2700,27780.6900,28328.1300,19804551031.2500
50%,58367.2100,59801.0700,57214.3100,58601.7200,30120046024.5000
75%,84754.2700,86483.9300,83567.9400,84752.1100,44062533458.5000
max,124752.1400,126198.0700,123196.0500,124752.5300,181746419401.0000


---
## Sección 3: Estadísticas Descriptivas Comparativas

Calculamos indicadores clave para comparar todos los activos entre sí.

### Métricas calculadas:
- **Volatilidad Anual**: Desviación estándar de retornos diarios × √252 (días hábiles). Mayor = más riesgo.
- **Sesgo (Skewness)**: 0 = simétrico, negativo = más pérdidas extremas que ganancias.
- **Curtosis (Kurtosis)**: 0 = normal, positivo = colas gruesas (eventos extremos frecuentes).
- **Ratio de Sharpe**: Retorno / Riesgo. Mayor = más eficiente.
- **Max Drawdown**: Pérdida máxima desde un máximo previo. Mide el peor escenario histórico.

In [4]:
# ============================================================
# CELDA 4: Tabla comparativa de estadísticas de todos los activos
# ============================================================

stats_list = []

for ticker, df in data.items():
    if 'Close' not in df.columns:
        continue
    close = df['Close'].dropna()
    
    # Retornos logarítmicos diarios
    # ln(P_t / P_{t-1}) es más apropiado que retorno simple para análisis financiero
    # porque es aditivo en el tiempo y más simétrico
    log_ret = np.log(close / close.shift(1)).dropna()
    
    stats_list.append({
        'Activo'        : ticker,
        'Nombre'        : ACTIVOS[ticker]['nombre'],
        'Tipo'          : ACTIVOS[ticker]['tipo'],
        'Precio Min'    : round(close.min(), 2),
        'Precio Max'    : round(close.max(), 2),
        'Precio Actual' : round(close.iloc[-1], 2),
        'Ret. Diario %' : round(log_ret.mean() * 100, 4),
        'Vol. Anual %'  : round(log_ret.std() * np.sqrt(252) * 100, 2),
        'Sesgo'         : round(log_ret.skew(), 3),
        'Curtosis'      : round(log_ret.kurt(), 3),
        'Sharpe Anual'  : round((log_ret.mean() / log_ret.std()) * np.sqrt(252), 3),
        'Max Drawdown %': round(((close / close.cummax()) - 1).min() * 100, 2),
    })

stats_df = pd.DataFrame(stats_list).set_index('Activo')

print('=== ESTADISTICAS COMPARATIVAS DE ACTIVOS ===')
print('    Periodo: Enero 2022 - Junio 2026')
print()
display(stats_df)

print()
print('INTERPRETACION:')
print('  - Vol. Anual %  : BTC/ETH (~70-80%) son mucho mas volatiles que SPY (~15%)')
print('  - Sesgo         : Negativo en cripto -> mas caidas extremas que subidas')
print('  - Curtosis      : Alta en cripto -> colas gruesas -> eventos extremos frecuentes')
print('  - Sharpe Anual  : Retorno por unidad de riesgo asumido')
print('  - Max Drawdown  : BTC puede caer >70% desde maximos historicos')
print()
print('RELEVANCIA PARA EL TFM:')
print('  La alta curtosis y sesgo negativo de cripto indican que los eventos extremos')
print('  ocurren mas frecuentemente de lo esperado por la teoria clasica.')
print('  HIPOTESIS: Estos eventos extremos estan impulsados por el sentimiento del mercado,')
print('  lo que justifica incorporar el analisis de sentimiento al modelo predictivo.')

=== ESTADISTICAS COMPARATIVAS DE ACTIVOS ===
    Periodo: Enero 2022 - Junio 2026



,Nombre,Tipo,Precio Min,Precio Max,Precio Actual,Ret. Diario %,Vol. Anual %,Sesgo,Curtosis,Sharpe Anual,Max Drawdown %
Activo,,,,,,,,,,,
BTC-USD,Bitcoin,Criptomoneda,15787.2800,124752.5300,64239.6800,0.0183,42.7300,-0.1890,4.7880,0.1080,-66.8900
ETH-USD,Ethereum,Criptomoneda,993.6400,4831.3500,1739.3000,-0.0474,57.7700,-0.0740,4.0470,-0.2070,-74.0500
SPY,S&P 500 ETF,ETF Global,339.3800,757.6200,746.7400,0.0454,17.6200,0.1670,7.9440,0.6490,-24.5000
EEM,Mercados Emergentes,ETF Regional,30.8800,70.7900,70.7900,0.0417,19.7500,0.1280,3.8420,0.5330,-32.7100
ILF,Latinoamerica 40 ETF,ETF Latam,16.3400,37.6200,33.9000,0.0596,23.1400,-0.1830,1.5350,0.6490,-29.7100
EWZ,Brasil ETF,ETF Brasil,19.3100,41.3400,33.7300,0.0471,27.3200,-0.2560,1.8770,0.4340,-32.2400
GLD,Oro (Gold ETF),Commodity,151.2300,495.9000,387.1200,0.0745,18.9100,-0.8890,8.4620,0.9930,-24.4600
USO,Petroleo (Oil ETF),Commodity,54.8300,152.9600,114.8700,0.0662,36.7200,-0.2330,2.8470,0.4540,-36.2300



INTERPRETACION:
  - Vol. Anual %  : BTC/ETH (~70-80%) son mucho mas volatiles que SPY (~15%)
  - Sesgo         : Negativo en cripto -> mas caidas extremas que subidas
  - Curtosis      : Alta en cripto -> colas gruesas -> eventos extremos frecuentes
  - Sharpe Anual  : Retorno por unidad de riesgo asumido
  - Max Drawdown  : BTC puede caer >70% desde maximos historicos

RELEVANCIA PARA EL TFM:
  La alta curtosis y sesgo negativo de cripto indican que los eventos extremos
  ocurren mas frecuentemente de lo esperado por la teoria clasica.
  HIPOTESIS: Estos eventos extremos estan impulsados por el sentimiento del mercado,
  lo que justifica incorporar el analisis de sentimiento al modelo predictivo.


---
## Sección 4: Visualización de Series Temporales

Visualizamos la evolución del precio de cada activo desde 2022, normalizados a base 100.

**¿Por qué normalizar?** Los activos tienen precios en escalas muy distintas (BTC ~$60,000 vs ETH ~$3,000 vs SPY ~$500). Al normalizar todos a base 100 (= precio al inicio del período), podemos comparar el **rendimiento relativo** de cada activo en el mismo gráfico.

**Eventos marcados:**
- **Mayo 2022**: Colapso de Terra/LUNA — pérdida de $60B en días. Generó pánico masivo en redes.
- **Nov. 2022**: Colapso de FTX (exchange cripto) — fraude de $8B. Uno de los peores momentos del mercado cripto.
- **Ene. 2024**: Aprobación del primer ETF de Bitcoin al contado en EEUU — evento muy positivo para el mercado.
- **Abr. 2024**: Bitcoin Halving — evento programado cada 4 años que reduce a la mitad la emisión de BTC.

In [5]:
# ============================================================
# CELDA 5: Grafico 1 — Series temporales normalizadas a base 100
# ============================================================
# Base 100 significa: si el activo estaba en 100 al inicio,
# un valor de 200 significa que doblo su precio
# un valor de 50  significa que perdio la mitad de su valor

fig, axes = plt.subplots(2, 1, figsize=(16, 11))
fig.suptitle('Evolución de Precios Normalizados (Base 100 = Enero 2022)\n'
             'Mercados Financieros Relevantes para el Análisis de Sentimiento en Ecuador',
             fontsize=14, fontweight='bold', y=1.01)

# ── Panel superior: Criptomonedas ────────────────────────────
ax1 = axes[0]
for ticker in ['BTC-USD', 'ETH-USD']:
    if ticker in data:
        close = data[ticker]['Close'].dropna()
        normalizado = (close / close.iloc[0]) * 100
        ax1.plot(normalizado.index, normalizado.values,
                 label=f"{ACTIVOS[ticker]['nombre']} ({ticker})",
                 color=ACTIVOS[ticker]['color'], linewidth=1.8, alpha=0.9)

# Marcar eventos históricos clave (estos momentos generaron el mayor sentimiento)
eventos_cripto = [
    ('2022-05-09', 'Colapso\nTerra/LUNA', 'red'),
    ('2022-11-08', 'Colapso\nFTX', 'red'),
    ('2024-01-11', 'ETF BTC\naprobado', 'green'),
    ('2024-04-19', 'Bitcoin\nHalving', 'blue'),
]
ymax = 450
for fecha_str, texto, color_ev in eventos_cripto:
    dt = pd.to_datetime(fecha_str)
    ax1.axvline(x=dt, color=color_ev, linestyle='--', alpha=0.5, linewidth=1.2)
    ax1.text(dt, ymax, texto, fontsize=7.5, color=color_ev, ha='center',
             bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor=color_ev))

ax1.axhline(y=100, color='gray', linestyle='-', alpha=0.4, linewidth=1)
ax1.set_title('Criptomonedas — Alta sensibilidad al sentimiento en redes sociales', fontsize=12)
ax1.set_ylabel('Precio relativo (Base 100)', fontsize=11)
ax1.legend(loc='upper left', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, ymax + 30)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)

# ── Panel inferior: ETFs y Commodities ───────────────────────
ax2 = axes[1]
for ticker in ['SPY', 'EEM', 'ILF', 'GLD', 'USO']:
    if ticker in data:
        close = data[ticker]['Close'].dropna()
        normalizado = (close / close.iloc[0]) * 100
        ax2.plot(normalizado.index, normalizado.values,
                 label=ACTIVOS[ticker]['nombre'],
                 color=ACTIVOS[ticker]['color'], linewidth=1.8, alpha=0.9)

ax2.axhline(y=100, color='gray', linestyle='-', alpha=0.4, linewidth=1)
ax2.set_title('ETFs y Commodities — Relevantes para la economía de Ecuador (dolarizada)', fontsize=12)
ax2.set_ylabel('Precio relativo (Base 100)', fontsize=11)
ax2.set_xlabel('Fecha', fontsize=11)
ax2.legend(loc='upper left', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
out1 = FIGURES_DIR / '01_series_temporales_normalizadas.png'
plt.savefig(out1, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out1.name}')

Guardado: 01_series_temporales_normalizadas.png


In [6]:
# ============================================================
# CELDA 6: Grafico 2 — Retornos diarios y volatilidad
# ============================================================
# Los RETORNOS son la variación porcentual diaria del precio.
# Formula: Retorno_t = ln(Precio_t / Precio_{t-1}) * 100
#
# Este gráfico muestra dos cosas clave:
# 1. La magnitud de los movimientos diarios (barras)
# 2. La volatilidad dinámica (linea naranja = std movil 30 dias)
#
# HIPOTESIS DEL TFM: Los periodos de alta volatilidad (linea naranja alta)
# coincidiran con periodos de alto volumen de sentiment en redes sociales.

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
fig.suptitle('Retornos Logaritmicos Diarios — Evidencia de Clustering de Volatilidad',
             fontsize=14, fontweight='bold')

for i, ticker in enumerate(['BTC-USD', 'ETH-USD']):
    if ticker not in data:
        continue
    close  = data[ticker]['Close'].dropna()
    retorn = np.log(close / close.shift(1)).dropna() * 100
    vol30  = retorn.rolling(30).std()

    ax = axes[i]
    colors_bar = ['#cc0000' if r < 0 else '#00aa00' for r in retorn]
    ax.bar(retorn.index, retorn.values, color=colors_bar, alpha=0.65, width=1)
    ax.axhline(y=0, color='black', linewidth=0.8)

    # Banda de volatilidad (1 desviacion estandar movil 30 dias)
    ax.plot(vol30.index, vol30.values,  color='orange', linewidth=1.8, label='Volatilidad 30d (1-sigma)', zorder=3)
    ax.plot(vol30.index, -vol30.values, color='orange', linewidth=1.8, zorder=3)
    ax.fill_between(vol30.index, vol30.values, -vol30.values, alpha=0.08, color='orange')

    ax.set_title(f'Retornos Diarios — {ACTIVOS[ticker]["nombre"]} ({ticker})', fontsize=12)
    ax.set_ylabel('Retorno (%)', fontsize=10)
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.3)

    # Estadisticas resumidas en el grafico
    info_txt = (f'Media: {retorn.mean():.3f}%  |  '
                f'Std: {retorn.std():.3f}%  |  '
                f'Peor dia: {retorn.min():.2f}%  |  '
                f'Mejor dia: {retorn.max():.2f}%')
    ax.text(0.01, 0.04, info_txt, transform=ax.transAxes, fontsize=8,
            verticalalignment='bottom',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.85))

axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)
axes[1].set_xlabel('Fecha', fontsize=11)

plt.tight_layout()
out2 = FIGURES_DIR / '02_retornos_diarios.png'
plt.savefig(out2, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out2.name}')
print()
print('INTERPRETACION:')
print('  Barras rojas    = dias de caida de precio')
print('  Barras verdes   = dias de subida de precio')
print('  Linea naranja   = volatilidad movil: cuando sube, el mercado esta mas incierto')
print('  HIPOTESIS: Los picos de volatilidad coincidiran con peaks de sentiment negativo')

Guardado: 02_retornos_diarios.png

INTERPRETACION:
  Barras rojas    = dias de caida de precio
  Barras verdes   = dias de subida de precio
  Linea naranja   = volatilidad movil: cuando sube, el mercado esta mas incierto
  HIPOTESIS: Los picos de volatilidad coincidiran con peaks de sentiment negativo


---
## Sección 5: Análisis de Correlación

La correlación entre activos es fundamental para el diseño del sistema de sentimiento.

**Interpretación:**
- Valor cercano a +1.0 → Los activos suben y bajan juntos (correlación positiva alta)
- Valor cercano a 0.0 → Los activos se mueven de forma independiente
- Valor cercano a -1.0 → Cuando uno sube, el otro baja (correlación negativa)

**Importancia para el TFM:** Si BTC y ETH tienen correlación de +0.85, significa que una noticia de sentimiento negativo sobre criptomonedas en general afectará a ambos de forma similar. Esto simplifica el sistema: un solo índice de sentimiento cripto puede cubrir ambos activos.

In [7]:
# ============================================================
# CELDA 7: Grafico 3 — Matriz de correlacion entre retornos
# ============================================================
# Usamos RETORNOS (no precios) para la correlacion.
# Los retornos son mas apropiados porque eliminan la tendencia
# y muestran la relacion real entre los movimientos diarios.

# Construir DataFrame consolidado de retornos
returns_dict = {}
nombres_cortos = {
    'BTC-USD': 'Bitcoin', 'ETH-USD': 'Ethereum',
    'SPY': 'S&P 500',   'EEM': 'Emergentes',
    'ILF': 'Latam',     'EWZ': 'Brasil',
    'GLD': 'Oro',       'USO': 'Petroleo'
}

for ticker, df in data.items():
    if 'Close' in df.columns:
        close = df['Close'].dropna()
        nombre = nombres_cortos.get(ticker, ticker)
        returns_dict[nombre] = np.log(close / close.shift(1))

returns_df  = pd.DataFrame(returns_dict).dropna()
corr_matrix = returns_df.corr()

# Grafico: heatmap de correlacion
fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',
    cmap='RdYlGn', vmin=-1, vmax=1, center=0,
    square=True, linewidths=0.5, ax=ax,
    annot_kws={'size': 11, 'weight': 'bold'},
    cbar_kws={'label': 'Correlacion de Pearson'}
)

ax.set_title('Matriz de Correlacion de Retornos Diarios (2022-2026)\n'
             'Verde = correlacion positiva  |  Rojo = correlacion negativa',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

plt.tight_layout()
out3 = FIGURES_DIR / '03_matriz_correlacion.png'
plt.savefig(out3, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out3.name}')

# Imprimir correlaciones más destacadas
print()
print('CORRELACIONES MAS ALTAS (>0.50 o <-0.50):')
pares_vistos = set()
for col1 in corr_matrix.columns:
    for col2 in corr_matrix.columns:
        if col1 != col2 and (col2, col1) not in pares_vistos:
            val = corr_matrix.loc[col1, col2]
            if abs(val) > 0.50:
                nivel = 'ALTA' if abs(val) > 0.75 else 'MODERADA'
                signo = 'POSITIVA' if val > 0 else 'NEGATIVA'
                print(f'  {col1:12s} <-> {col2:12s}: {val:.3f}  [{nivel} {signo}]')
            pares_vistos.add((col1, col2))

Guardado: 03_matriz_correlacion.png

CORRELACIONES MAS ALTAS (>0.50 o <-0.50):
  Bitcoin      <-> Ethereum    : 0.850  [ALTA POSITIVA]
  S&P 500      <-> Emergentes  : 0.690  [MODERADA POSITIVA]
  S&P 500      <-> Latam       : 0.540  [MODERADA POSITIVA]
  Emergentes   <-> Latam       : 0.661  [MODERADA POSITIVA]
  Emergentes   <-> Brasil      : 0.550  [MODERADA POSITIVA]
  Latam        <-> Brasil      : 0.942  [ALTA POSITIVA]


In [8]:
# ============================================================
# CELDA 8: Grafico 4 — Correlacion rodante (rolling correlation)
# ============================================================
# La correlacion NO es constante en el tiempo.
# En periodos de crisis, todos los activos tienden a caer juntos
# (correlacion sube hacia +1). Esto se llama "contagio financiero".
#
# Este fenomeno es clave para el TFM:
# El sentimiento negativo masivo es el mecanismo que SINCRONIZA los mercados.

VENTANA = 60  # Ventana de 60 dias para calcular la correlacion movil

fig, ax = plt.subplots(figsize=(16, 6))

btc_ret = returns_df['Bitcoin'].dropna()
comparar = [
    ('Ethereum',   '#627EEA'),
    ('S&P 500',    '#2196F3'),
    ('Emergentes', '#4CAF50'),
    ('Latam',      '#FF9800'),
    ('Petroleo',   '#795548'),
]

for activo, color in comparar:
    if activo not in returns_df.columns:
        continue
    alineado = pd.concat([btc_ret, returns_df[activo]], axis=1).dropna()
    corr_rol = alineado.iloc[:, 0].rolling(VENTANA).corr(alineado.iloc[:, 1])
    ax.plot(corr_rol.index, corr_rol.values,
            label=f'Bitcoin vs. {activo}', color=color, linewidth=1.5, alpha=0.85)

ax.axhline(y=0,    color='black', linewidth=0.8, linestyle='-')
ax.axhline(y=0.5,  color='gray',  linewidth=0.8, linestyle='--', alpha=0.5, label='Umbral +0.5')
ax.axhline(y=-0.5, color='gray',  linewidth=0.8, linestyle='--', alpha=0.5)

ax.set_title(f'Correlacion Rodante: Bitcoin vs. Otros Activos (Ventana = {VENTANA} dias)\n'
             'Picos de correlacion = momentos de contagio financiero por sentiment extremo',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Correlacion de Pearson', fontsize=11)
ax.set_xlabel('Fecha', fontsize=11)
ax.set_ylim(-1, 1)
ax.legend(loc='lower left', fontsize=9)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
out4 = FIGURES_DIR / '04_correlacion_rodante.png'
plt.savefig(out4, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out4.name}')

Guardado: 04_correlacion_rodante.png


---
## Sección 6: Análisis de Drawdown

El **Drawdown** mide cuánto ha caído un activo desde su máximo histórico previo.

**Fórmula:** `Drawdown_t = (Precio_t / max(Precio_0..Precio_t)) - 1`

Los períodos de mayor drawdown son los más importantes para el análisis de sentimiento, porque son exactamente los momentos donde el **pánico colectivo** se dispara en redes sociales y noticias, generando el mayor volumen de contenido con sentimiento negativo.

In [9]:
# ============================================================
# CELDA 9: Grafico 5 — Analisis de drawdown de criptomonedas
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
fig.suptitle('Analisis de Drawdown — Identificacion de Periodos Criticos\n'
             'Estos periodos generaron el mayor sentiment negativo en redes sociales',
             fontsize=13, fontweight='bold')

for i, ticker in enumerate(['BTC-USD', 'ETH-USD']):
    if ticker not in data:
        continue
    close = data[ticker]['Close'].dropna()
    dd    = (close / close.cummax() - 1) * 100  # Drawdown en %

    ax = axes[i]
    ax.fill_between(dd.index, dd.values, 0, color=ACTIVOS[ticker]['color'], alpha=0.4)
    ax.plot(dd.index, dd.values, color=ACTIVOS[ticker]['color'], linewidth=1)
    ax.axhline(y=0,   color='black', linewidth=0.8)
    ax.axhline(y=-20, color='orange', linewidth=0.8, linestyle='--', alpha=0.6, label='-20% umbral')
    ax.axhline(y=-50, color='red',    linewidth=0.8, linestyle='--', alpha=0.6, label='-50% umbral')

    # Anotar el peor drawdown del historico
    idx_peor = dd.idxmin()
    val_peor = dd.min()
    offset_x = pd.Timedelta(days=120)
    ax.annotate(
        f'Peor caida: {val_peor:.1f}%\n({idx_peor.strftime("%b %Y")})',
        xy=(idx_peor, val_peor),
        xytext=(idx_peor + offset_x, val_peor + 10),
        arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
        fontsize=9, color='red',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9, edgecolor='red')
    )

    ax.set_ylabel('Drawdown (%)', fontsize=10)
    ax.set_title(f'Drawdown — {ACTIVOS[ticker]["nombre"]}', fontsize=12)
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(val_peor * 1.1, 5)

# Panel 3: Retorno acumulado comparativo
ax3 = axes[2]
for ticker in ['BTC-USD', 'ETH-USD', 'SPY', 'GLD']:
    if ticker in data:
        close    = data[ticker]['Close'].dropna()
        ret_acum = ((close / close.iloc[0]) - 1) * 100
        ax3.plot(ret_acum.index, ret_acum.values,
                 label=ACTIVOS[ticker]['nombre'],
                 color=ACTIVOS[ticker]['color'], linewidth=1.5)

ax3.axhline(y=0, color='black', linewidth=0.8)
ax3.set_title('Retorno Acumulado desde Enero 2022 — Comparativa de Activos', fontsize=12)
ax3.set_ylabel('Retorno acumulado (%)', fontsize=10)
ax3.set_xlabel('Fecha', fontsize=11)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax3.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
out5 = FIGURES_DIR / '05_drawdown_analysis.png'
plt.savefig(out5, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out5.name}')

Guardado: 05_drawdown_analysis.png


---
## Sección 7: Distribución de Retornos y Prueba de Normalidad

### La teoría clásica asume distribución normal — los datos muestran lo contrario

Los modelos financieros clásicos (Black-Scholes, CAPM) asumen que los retornos siguen una distribución normal (Gaussiana). Esto implica que los eventos extremos son rarísimos.

**En la realidad**, los retornos financieros muestran:
- **Curtosis alta** (colas gruesas): Los crashes del -15% en un día ocurren con más frecuencia de lo que predice la curva normal.
- **Sesgo negativo**: Las caídas tienden a ser más bruscas que las subidas.

**¿Por qué ocurre esto?** Precisamente porque el **sentimiento colectivo** (pánico, euforia) genera movimientos que la lógica puramente racional no predice. Esto es la **justificación más fuerte** de nuestro TFM.

In [10]:
# ============================================================
# CELDA 10: Grafico 6 — Distribucion de retornos vs. Normal
# ============================================================
# El Test de Jarque-Bera verifica estadisticamente si los retornos
# son normales. Si p-valor < 0.05, rechazamos la hipotesis de normalidad.

btc_ret_pct = np.log(data['BTC-USD']['Close'] / data['BTC-USD']['Close'].shift(1)).dropna() * 100
eth_ret_pct = np.log(data['ETH-USD']['Close'] / data['ETH-USD']['Close'].shift(1)).dropna() * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Evidencia Empirica: Retornos Cripto NO siguen Distribucion Normal\n'
             'Justificacion de la necesidad del Analisis de Sentimiento',
             fontsize=13, fontweight='bold')

# --- Histograma ---
ax1 = axes[0]
ax1.hist(btc_ret_pct, bins=80, density=True, alpha=0.5, color='#F7931A',
         label='Bitcoin', edgecolor='none')
ax1.hist(eth_ret_pct, bins=80, density=True, alpha=0.5, color='#627EEA',
         label='Ethereum', edgecolor='none')

# Curva normal teorica para comparar
x_rng = np.linspace(min(btc_ret_pct.min(), eth_ret_pct.min()),
                    max(btc_ret_pct.max(), eth_ret_pct.max()), 300)
normal_curva = stats.norm.pdf(x_rng, btc_ret_pct.mean(), btc_ret_pct.std())
ax1.plot(x_rng, normal_curva, 'k--', linewidth=2, label='Normal teorica (BTC)')

ax1.set_title('Histograma de Retornos Diarios\nvs. Distribucion Normal Teorica', fontsize=12)
ax1.set_xlabel('Retorno diario (%)', fontsize=11)
ax1.set_ylabel('Densidad', fontsize=11)
ax1.set_xlim(-20, 20)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

info_txt = (f'BTC -> Sesgo: {btc_ret_pct.skew():.3f} | Curtosis: {btc_ret_pct.kurt():.3f}\n'
            f'ETH -> Sesgo: {eth_ret_pct.skew():.3f} | Curtosis: {eth_ret_pct.kurt():.3f}\n'
            f'(Distribucion Normal: Sesgo=0, Curtosis=0)')
ax1.text(0.02, 0.98, info_txt, transform=ax1.transAxes, va='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

# --- Q-Q Plot ---
ax2 = axes[1]
(osm, osr), (slope, intercept, _) = stats.probplot(btc_ret_pct, dist='norm')
ax2.scatter(osm, osr, s=4, alpha=0.35, color='#F7931A', label='Bitcoin')
(osm2, osr2), (slope2, intercept2, _) = stats.probplot(eth_ret_pct, dist='norm')
ax2.scatter(osm2, osr2, s=4, alpha=0.35, color='#627EEA', label='Ethereum')

x_qq = np.array([min(osm.min(), osm2.min()), max(osm.max(), osm2.max())])
ax2.plot(x_qq, slope * x_qq + intercept, 'k--', linewidth=2, label='Si fuera normal')
ax2.set_title('Grafico Q-Q — Desviacion de la Normalidad\n'
              'Las colas (extremos) se alejan de la linea diagonal', fontsize=12)
ax2.set_xlabel('Cuantiles teoricos (Normal)', fontsize=11)
ax2.set_ylabel('Cuantiles observados', fontsize=11)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
out6 = FIGURES_DIR / '06_distribucion_retornos.png'
plt.savefig(out6, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out6.name}')

# Test estadístico de normalidad
jb_btc = stats.jarque_bera(btc_ret_pct)
jb_eth = stats.jarque_bera(eth_ret_pct)
print()
print('TEST DE NORMALIDAD JARQUE-BERA (p-valor < 0.05 -> NO es normal):')
print(f'  Bitcoin  -> estadistico={jb_btc.statistic:.1f}, p-valor={jb_btc.pvalue:.2e}  => {"NO NORMAL" if jb_btc.pvalue < 0.05 else "Normal"}')
print(f'  Ethereum -> estadistico={jb_eth.statistic:.1f}, p-valor={jb_eth.pvalue:.2e}  => {"NO NORMAL" if jb_eth.pvalue < 0.05 else "Normal"}')
print()
print('CONCLUSION PARA EL TFM:')
print('  Los retornos NO son normales => hay mas eventos extremos de los esperados')
print('  El sentimiento colectivo (panico/euforia) es la causa de esos eventos extremos')
print('  => Incorporar analisis de sentimiento MEJORARA la capacidad predictiva del modelo')

Guardado: 06_distribucion_retornos.png

TEST DE NORMALIDAD JARQUE-BERA (p-valor < 0.05 -> NO es normal):
  Bitcoin  -> estadistico=1555.9, p-valor=0.00e+00  => NO NORMAL
  Ethereum -> estadistico=1105.4, p-valor=9.07e-241  => NO NORMAL

CONCLUSION PARA EL TFM:
  Los retornos NO son normales => hay mas eventos extremos de los esperados
  El sentimiento colectivo (panico/euforia) es la causa de esos eventos extremos
  => Incorporar analisis de sentimiento MEJORARA la capacidad predictiva del modelo


---
## Sección 8: Construcción del Dataset Maestro

Creamos el archivo `master_prices.csv` que combina todos los precios con variables técnicas calculadas.
Este será la base que en el **Notebook 04** combinaremos con el Índice de Sentimiento (ISF).

In [11]:
# ============================================================
# CELDA 11: Construccion del Dataset Maestro de Precios
# ============================================================
# Variables que calculamos:
#   _ret     : Retorno logaritmico diario
#   _vol14d  : Volatilidad anualizada con ventana de 14 dias
#   _sma7    : Media movil simple de 7 dias (tendencia corto plazo)
#   _sma21   : Media movil simple de 21 dias (tendencia medio plazo)
#   BTC_RSI14: Relative Strength Index de BTC (indica sobrecompra/sobreventa)
#   ETH_RSI14: Relative Strength Index de ETH

# Precio de cierre de todos los activos
close_prices = pd.DataFrame()
for ticker, df in data.items():
    if 'Close' in df.columns:
        close_prices[ticker] = df['Close']

# Retornos logaritmicos diarios
log_ret = np.log(close_prices / close_prices.shift(1))
log_ret.columns = [f'{c}_ret' for c in log_ret.columns]

# Volatilidad anualizada 14 dias
vol14 = log_ret.rolling(14).std() * np.sqrt(252) * 100
vol14.columns = [f'{c.replace("_ret","")}_vol14d' for c in vol14.columns]

# Medias moviles simples
sma7  = close_prices.rolling(7).mean()
sma7.columns  = [f'{c}_sma7'  for c in sma7.columns]
sma21 = close_prices.rolling(21).mean()
sma21.columns = [f'{c}_sma21' for c in sma21.columns]

# RSI (Relative Strength Index) - indicador tecnico clasico
def rsi(serie, period=14):
    """RSI: mide momentum. >70=sobrecomprado (posible caida), <30=sobrevendido (posible rebote)."""
    delta = serie.diff()
    gain  = delta.where(delta > 0, 0).rolling(period).mean()
    loss  = (-delta.where(delta < 0, 0)).rolling(period).mean()
    rs    = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

rsi_btc = rsi(close_prices['BTC-USD']).rename('BTC_RSI14')
rsi_eth = rsi(close_prices['ETH-USD']).rename('ETH_RSI14')

# Combinar todo en DataFrame maestro
master_df = pd.concat([close_prices, log_ret, vol14, sma7, sma21, rsi_btc, rsi_eth], axis=1)

# Eliminar filas con demasiados nulos (primeros registros por medias moviles)
master_df = master_df.dropna(thresh=int(len(master_df.columns) * 0.65))

# Guardar
out_master = PROCESSED_DIR / 'master_prices.csv'
master_df.to_csv(out_master)

print('=== DATASET MAESTRO DE PRECIOS CREADO ===')
print(f'  Dimensiones : {master_df.shape[0]:,} filas x {master_df.shape[1]} columnas')
print(f'  Periodo     : {master_df.index.min().date()} -> {master_df.index.max().date()}')
print(f'  Archivo     : {out_master}')
print()
print('Columnas incluidas:')
for i, col in enumerate(master_df.columns, 1):
    print(f'  {i:2d}. {col}')

print()
print('Vista previa de las ultimas 3 filas (columnas clave):')
cols_preview = ['BTC-USD', 'ETH-USD', 'BTC-USD_ret', 'ETH-USD_ret', 'BTC-USD_vol14d', 'BTC_RSI14']
display(master_df[[c for c in cols_preview if c in master_df.columns]].tail(3).round(4))

=== DATASET MAESTRO DE PRECIOS CREADO ===
  Dimensiones : 0 filas x 42 columnas
  Periodo     : NaT -> NaT
  Archivo     : C:\Users\StivArmas\OneDrive - ROBALINO\Documentos\MASTER\TFM\tfm-sentiment-financial\data\processed\master_prices.csv

Columnas incluidas:
   1. BTC-USD
   2. ETH-USD
   3. SPY
   4. EEM
   5. ILF
   6. EWZ
   7. GLD
   8. USO
   9. BTC-USD_ret
  10. ETH-USD_ret
  11. SPY_ret
  12. EEM_ret
  13. ILF_ret
  14. EWZ_ret
  15. GLD_ret
  16. USO_ret
  17. BTC-USD_vol14d
  18. ETH-USD_vol14d
  19. SPY_vol14d
  20. EEM_vol14d
  21. ILF_vol14d
  22. EWZ_vol14d
  23. GLD_vol14d
  24. USO_vol14d
  25. BTC-USD_sma7
  26. ETH-USD_sma7
  27. SPY_sma7
  28. EEM_sma7
  29. ILF_sma7
  30. EWZ_sma7
  31. GLD_sma7
  32. USO_sma7
  33. BTC-USD_sma21
  34. ETH-USD_sma21
  35. SPY_sma21
  36. EEM_sma21
  37. ILF_sma21
  38. EWZ_sma21
  39. GLD_sma21
  40. USO_sma21
  41. BTC_RSI14
  42. ETH_RSI14

Vista previa de las ultimas 3 filas (columnas clave):


,BTC-USD,ETH-USD,BTC-USD_ret,ETH-USD_ret,BTC-USD_vol14d,BTC_RSI14
Date,,,,,,


In [12]:
# ============================================================
# CELDA 12: Grafico 7 — BTC con indicadores tecnicos
# ============================================================
# Esta es la visualizacion final del notebook.
# En el Notebook 04 añadiremos el Indice de Sentimiento (ISF)
# sobre este mismo grafico para mostrar la correlacion visual.

fecha_inicio = pd.Timestamp.now() - pd.DateOffset(months=18)
subset = master_df.loc[master_df.index >= fecha_inicio]

fig, axes = plt.subplots(3, 1, figsize=(16, 13),
                         gridspec_kw={'height_ratios': [3, 1.2, 1.2]},
                         sharex=True)
fig.suptitle('Bitcoin — Analisis Tecnico (Ultimos 18 meses)\n'
             'Base para la futura integracion con el Indice de Sentimiento (ISF)',
             fontsize=13, fontweight='bold')

# Panel 1: Precio + Medias Moviles
ax1 = axes[0]
ax1.plot(subset.index, subset['BTC-USD'],
         color='#F7931A', linewidth=2.2, label='Precio BTC', alpha=0.95, zorder=3)
if 'BTC-USD_sma7'  in subset: ax1.plot(subset.index, subset['BTC-USD_sma7'],  'b--', lw=1.2, label='SMA 7d',  alpha=0.8)
if 'BTC-USD_sma21' in subset: ax1.plot(subset.index, subset['BTC-USD_sma21'], 'm-.', lw=1.2, label='SMA 21d', alpha=0.8)

ax1.set_ylabel('Precio BTC (USD)', fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Panel 2: RSI
ax2 = axes[1]
if 'BTC_RSI14' in subset:
    rsi_s = subset['BTC_RSI14'].dropna()
    ax2.plot(rsi_s.index, rsi_s.values, color='#9C27B0', linewidth=1.4)
    ax2.fill_between(rsi_s.index, 70, rsi_s.values, where=(rsi_s.values>70), color='red',   alpha=0.25, label='Sobrecomprado (>70)')
    ax2.fill_between(rsi_s.index, rsi_s.values, 30, where=(rsi_s.values<30), color='green', alpha=0.25, label='Sobrevendido (<30)')
    ax2.axhline(y=70, color='red',   linestyle='--', linewidth=0.9, alpha=0.7)
    ax2.axhline(y=30, color='green', linestyle='--', linewidth=0.9, alpha=0.7)
    ax2.axhline(y=50, color='gray',  linestyle='-',  linewidth=0.5, alpha=0.5)
    ax2.set_ylabel('RSI (14d)', fontsize=10)
    ax2.set_ylim(0, 100)
    ax2.legend(fontsize=8, loc='upper left')
    ax2.grid(True, alpha=0.3)

# Panel 3: Volatilidad
ax3 = axes[2]
if 'BTC-USD_vol14d' in subset:
    vol_s = subset['BTC-USD_vol14d'].dropna()
    ax3.fill_between(vol_s.index, vol_s.values, alpha=0.45, color='#FF5722')
    ax3.plot(vol_s.index, vol_s.values, color='#FF5722', linewidth=1.2)
    ax3.set_ylabel('Volatilidad\nAnualiz. (%)', fontsize=9)
    ax3.set_xlabel('Fecha', fontsize=11)
    ax3.grid(True, alpha=0.3)

ax3.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax3.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,3,5,7,9,11]))
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
out7 = FIGURES_DIR / '07_btc_indicadores_tecnicos.png'
plt.savefig(out7, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out7.name}')

Guardado: 07_btc_indicadores_tecnicos.png


---
## Sección 9: Resumen y Conclusiones del Notebook 01

### Lo que hemos hecho:

| Paso | Resultado |
|---|---|
| Carga de datos | 8 activos, ~1,600+ registros diarios c/u (2022–2026) |
| Estadísticas descriptivas | Tabla comparativa de volatilidad, Sharpe, drawdown |
| Series temporales | Precios normalizados base 100 con eventos históricos |
| Análisis de retornos | Distribución diaria y clustering de volatilidad |
| Correlación | Matriz estática + rolling correlation 60 días |
| Drawdown | Períodos críticos identificados |
| Normalidad | Test Jarque-Bera confirmó NO normalidad |
| Dataset maestro | `data/processed/master_prices.csv` con 30+ variables |

### Hallazgos clave para la tesis:

1. **BTC y ETH** tienen correlación alta entre sí (>0.85) → el sentimiento cripto les afecta de forma similar
2. **Los retornos NO son normales** → los modelos clásicos fallan → necesitamos capturar el sentimiento
3. **Los períodos de mayor drawdown** coincidirán con los de mayor sentimiento negativo en redes
4. **La correlación BTC-SPY aumenta en crisis** → el pánico del mercado global afecta incluso a Ecuador

### Siguiente Notebook:
**Notebook 02 — Preprocesamiento NLP + EDA de Textos**: Limpieza y análisis de las noticias ecuatorianas y tweets recolectados.

In [13]:
# ============================================================
# CELDA FINAL: Verificacion de todos los archivos generados
# ============================================================

print('=' * 60)
print('  VERIFICACION FINAL — Notebook 01 Completado')
print('=' * 60)

archivos = [
    (PROCESSED_DIR / 'master_prices.csv',                     'Dataset maestro de precios'),
    (FIGURES_DIR   / '01_series_temporales_normalizadas.png', 'Series temporales normalizadas'),
    (FIGURES_DIR   / '02_retornos_diarios.png',               'Retornos y volatilidad diaria'),
    (FIGURES_DIR   / '03_matriz_correlacion.png',             'Matriz de correlacion'),
    (FIGURES_DIR   / '04_correlacion_rodante.png',            'Correlacion rodante 60d'),
    (FIGURES_DIR   / '05_drawdown_analysis.png',              'Analisis de drawdown'),
    (FIGURES_DIR   / '06_distribucion_retornos.png',          'Distribucion de retornos'),
    (FIGURES_DIR   / '07_btc_indicadores_tecnicos.png',       'BTC con indicadores tecnicos'),
]

print()
todos_ok = True
for ruta, desc in archivos:
    if ruta.exists():
        size_kb = ruta.stat().st_size / 1024
        print(f'  OK  {ruta.name:45s} {size_kb:7.1f} KB  - {desc}')
    else:
        print(f'  NO  {ruta.name:45s}  NO ENCONTRADO  - {desc}')
        todos_ok = False

print()
if todos_ok:
    print('  Todos los archivos generados correctamente.')
    print()
    print('  Siguiente paso: Abrir notebooks/02_eda_preprocessing.ipynb')
else:
    print('  ATENCION: Algunos archivos no se generaron. Revisar celdas anteriores.')

  VERIFICACION FINAL — Notebook 01 Completado

  OK  master_prices.csv                                 0.4 KB  - Dataset maestro de precios
  OK  01_series_temporales_normalizadas.png           475.5 KB  - Series temporales normalizadas
  OK  02_retornos_diarios.png                         262.2 KB  - Retornos y volatilidad diaria
  OK  03_matriz_correlacion.png                       140.7 KB  - Matriz de correlacion
  OK  04_correlacion_rodante.png                      334.3 KB  - Correlacion rodante 60d
  OK  05_drawdown_analysis.png                        496.9 KB  - Analisis de drawdown
  OK  06_distribucion_retornos.png                    168.3 KB  - Distribucion de retornos
  OK  07_btc_indicadores_tecnicos.png                  88.7 KB  - BTC con indicadores tecnicos

  Todos los archivos generados correctamente.

  Siguiente paso: Abrir notebooks/02_eda_preprocessing.ipynb
